In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from tti.traveltimes.paths import (
    calculate_path_direction_vector,
    spherical_to_cartesian,
)

In [ ]:
R = 1221.5

In [ ]:
polar_path = np.array([[0.0, -90.0, R], [0.0, 90.0, R]])
diag_path = np.array([[-135.0, -45.0, R], [135.0, 45.0, R]])
equator_path = np.array([[0.0, 0.0, R], [120.0, 0.0, R]])
tilted_path = np.array([[45.0, -30.0, R], [225.0, 30.0, R]])
offset_path = np.array([[10.0, -20.0, R], [190.0, 20.0, R]])
random_path = np.array([[30.0, -60.0, R], [-45.0, 10.0, R]])
nearby_path = np.array([[10.0, 10.0, R], [20.0, 15.0, R]])
long_but_not_antipodal = np.array([[0.0, 10.0, R], [160.0, 20.0, R]])
shallow_path = np.array([[-60.0, 30.0, R], [10.0, 35.0, R]])
small_sep_path = np.array([[100.0, -10.0, R], [110.0, -8.0, R]])

# Stack all inbound/outbound points. None of the above pairs are exact antipodes,
# so their chords do not pass through the origin (centre of the Earth).
ic_in = np.stack(
    [
        a[0]
        for a in [
            polar_path,
            diag_path,
            equator_path,
            tilted_path,
            offset_path,
            random_path,
            nearby_path,
            long_but_not_antipodal,
            shallow_path,
            small_sep_path,
        ]
    ]
)
ic_out = np.stack(
    [
        a[1]
        for a in [
            polar_path,
            diag_path,
            equator_path,
            tilted_path,
            offset_path,
            random_path,
            nearby_path,
            long_but_not_antipodal,
            shallow_path,
            small_sep_path,
        ]
    ]
)
path_directions = calculate_path_direction_vector(ic_in, ic_out)

ic_in_cartesian = spherical_to_cartesian(*ic_in.T)
ic_out_cartesian = spherical_to_cartesian(*ic_out.T)

Gradient of the distance travelled in the imic $w_1$ wrt imir radius $r$.

$$\frac{\partial w_1}{\partial r} = \frac{4r}{\sqrt{\det(r)}} $$

where $\mathbf{o}$ is the ray origin, $\hat{\mathbf{n}}$ is the path direction and $\det{r}$ is the determinant of the quadratic formula for finding the pierce points.

$$ \det(r) = (2\mathbf{o}\cdot\hat{\mathbf{n}})^2 - 4(\hat{\mathbf{n}}\cdot\hat{\mathbf{n}})(\mathbf{o}\cdot\mathbf{o} - r^2) $$

For paths that go straight through the centre of the Earth this reduces to
$$\frac{\partial w_1}{\partial r} = 2 $$

In [ ]:
def grad_w1(o, n, r):
    o_dot_n = np.sum(o * n, axis=-1, keepdims=True)
    o_dot_o = np.sum(o * o, axis=-1, keepdims=True)
    n_dot_n = np.sum(n * n, axis=-1, keepdims=True)

    a = n_dot_n
    b = 2 * o_dot_n
    c = o_dot_o - r**2

    det = b**2 - 4 * a * c
    det[det <= 0] = 0.0

    with np.errstate(
        divide="ignore", invalid="ignore"
    ):  # Ignore division by zero warnings
        gradient = 4 * r / np.sqrt(det)
    gradient[np.isnan(gradient) | np.isinf(gradient)] = (
        0.0  # when det <= 0, set gradient to 0
    )
    return gradient


r = np.linspace(0.0, R, 1000)
g = grad_w1(ic_in_cartesian, path_directions, r)
plt.plot(r, g.T)
plt.xlabel("IMIC Radius (km)")
plt.ylabel("Gradient")
plt.title("Gradient of distance travelled in IMIC with respect to IMIC Radius")

In [ ]:
from tti.elastic import transverse_isotropic_tensor
from tti.traveltimes.traveltimes import calculate_relative_traveltime_voigt

lam_1 = 3
mu_1 = 4
lam_2 = 2
mu_2 = 1

D_fast_imic = transverse_isotropic_tensor(
    np.array(
        [lam_1 + 2 * mu_1, lam_2 + 2 * mu_2]
    ),  # faster IMIC, as r increases more of the path is fast so tt decreases so gradient is negative
    np.array([lam_1 + 2 * mu_1, lam_2 + 2 * mu_2]),
    np.array([lam_1, lam_2]),
    np.array([mu_1, mu_2]),
    np.array([mu_1, mu_2]),
)
D_slow_imic = transverse_isotropic_tensor(
    np.array(
        [lam_2 + 2 * mu_2, lam_1 + 2 * mu_1]
    ),  # slower IMIC, as r increases more of the path is slow so tt increases so gradient is positive
    np.array([lam_2 + 2 * mu_2, lam_1 + 2 * mu_1]),
    np.array([lam_2, lam_1]),
    np.array([mu_2, mu_1]),
    np.array([mu_2, mu_1]),
)


def grad_t(o, n, r, D):
    dw1_dr = grad_w1(o, n, r)
    weights = np.stack([dw1_dr, -dw1_dr]).transpose(
        2, 0, 1
    )  # (batch, nsegments, npaths)
    dt_dr = np.sum(
        weights * calculate_relative_traveltime_voigt(n, D, normalisation=-1 / 2),
        axis=-2,
    )
    return dt_dr


axs = plt.figure(figsize=(12, 5)).subplots(1, 2)
for ax, D, title in zip(
    axs.flatten(), [D_fast_imic, D_slow_imic], ["Fast IMIC", "Slow IMIC"]
):
    g = grad_t(ic_in_cartesian, path_directions, r, D)
    ax.plot(r, g)
    ax.set_xlabel("IMIC Radius (km)")
    ax.set_title(title)

axs[0].set_ylabel("Gradient of travel time wrt IMIC Radius")
axs[1].set_yticklabels([])